# 부품 7종 YOLO11 검출 모델 — Colab GPU 학습

**시작:** Colab 메뉴 `런타임 → 런타임 유형 변경 → GPU` 선택 후 위에서 아래로 실행하세요.
`components_dataset.zip`을 준비해 주세요. ROS 설치는 필요 없습니다.

클래스: **DHT11 / bluetooth / led_green / led_red / led_yellow / rgb / usb**

- `electric bulb`와 기존 battery 클래스는 이 데이터에 포함하지 않았습니다.
- 빨강·초록·노랑 LED는 **플라스틱 몸체만** 라벨링했으며 금속 핀은 제외했습니다.
- RGB 모듈은 PCB와 LED 렌즈를 포함합니다.
- 306장 = 학습 243장 / 검증 63장. 후보 337장 중 31장을 제외했습니다.
- 유사 이미지 그룹은 같은 분할에 배치했습니다. 같은 배경/촬영 환경이므로 독립 테스트는 아닙니다.
- 이 노트북은 7개 클래스용 새 검출 모델을 학습합니다. 이전 3클래스 `best.pt`는 덮어쓰지 않습니다.

사용법: 설정 → 설치/GPU 확인 → ZIP 업로드 → 라벨 검사 → 미리보기 → 학습 → 평가/다운로드.


In [ ]:
# 원하는 모델 크기: "n"(가벼움), "s"(연산량 증가)
MODEL_SIZE = "n"  # @param ["n", "s"]
EPOCHS = 100  # @param {type:"integer"}
IMGSZ = 640  # @param [320, 416, 640] {type:"raw"}
PATIENCE = 20
SEED = 44
# 런타임 종료에 대비해 Google Drive에 학습 결과 저장 (권장).
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
assert MODEL_SIZE in ("n", "s")
assert EPOCHS > 0 and IMGSZ in (320, 416, 640)
MODEL_WEIGHTS = f"yolo11{MODEL_SIZE}.pt"
print("학습할 모델:", MODEL_WEIGHTS)


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["YOLO_CONFIG_DIR"] = "/content/yolo_config"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.183"])
import torch
import ultralytics
from ultralytics import YOLO

print("torch:", torch.__version__, "ultralytics:", ultralytics.__version__)
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
import zipfile
import tempfile

print("components_dataset.zip 하나를 선택하세요.")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(zip_names) == 1, "데이터 ZIP을 정확히 하나 업로드하세요."
archive = Path(zip_names[0])
EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="components_", dir="/content"))
with zipfile.ZipFile(archive) as z:
    assert len(z.infolist()) < 10000, "파일 수가 예상보다 많습니다."
    for entry in z.infolist():
        target = (EXTRACT_ROOT / entry.filename).resolve()
        assert target.is_relative_to(EXTRACT_ROOT.resolve()), "잘못된 ZIP 경로입니다."
    z.extractall(EXTRACT_ROOT)
DATASET = EXTRACT_ROOT / "components_dataset"
assert (DATASET / "data.yaml").is_file(), "components_dataset 폴더가 있는 ZIP인지 확인하세요."
del uploaded
print("데이터 위치:", DATASET)


In [ ]:
import json
import yaml
import hashlib
from collections import Counter, defaultdict
from PIL import Image

EXPECTED_CLASSES = ["DHT11", "bluetooth", "led_green", "led_red", "led_yellow", "rgb", "usb"]
config = yaml.safe_load((DATASET / "data.yaml").read_text())
assert config["names"] == dict(enumerate(EXPECTED_CLASSES)), "클래스 ID/이름이 예상과 다릅니다."
# 원래 노트북 경로 대신 Colab의 실제 압축 해제 경로로 변경합니다.
config["path"] = str(DATASET)
(DATASET / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
DATA_YAML = str(DATASET / "data.yaml")
counts = {}
hashes = defaultdict(set)
for split in ("train", "val"):
    counts[split] = Counter()
    images = sorted((DATASET / "images" / split).glob("*.jpg"))
    assert images, f"{split} 이미지가 없습니다."
    labels = list((DATASET / "labels" / split).glob("*.txt"))
    assert len(images) == len(labels), "이미지/라벨 수가 다릅니다."
    for image_path in images:
        with Image.open(image_path) as image:
            image.verify()
        label = DATASET / "labels" / split / (image_path.stem + ".txt")
        assert label.is_file(), f"라벨 누락: {image_path}"
        rows = label.read_text().strip().splitlines()
        assert rows, "이 패키지의 채택 이미지에는 객체 라벨이 있어야 합니다."
        for row in rows:
            parts = row.split()
            assert len(parts) == 5
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            assert 0 <= class_id < 7
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
            assert x-w/2 >= -1e-6 and y-h/2 >= -1e-6
            assert x+w/2 <= 1+1e-6 and y+h/2 <= 1+1e-6
            counts[split][class_id] += 1
        hashes[hashlib.sha256(image_path.read_bytes()).hexdigest()].add(split)
    assert set(counts[split]) == set(range(7)), f"{split}에 일부 클래스가 없습니다."
assert all(len(splits) == 1 for splits in hashes.values()), "동일 이미지가 학습/검증에 중복됐습니다."
manifest = json.loads((DATASET / "manifest.json").read_text())
groups = defaultdict(set)
for item in manifest:
    if item["status"] == "included":
        assert item["reviewed"]
        groups[item["similarity_group"]].add(item["split"])
assert all(len(splits) == 1 for splits in groups.values())
for i, name in enumerate(EXPECTED_CLASSES):
    print(f"{i}: {name:12s} train={counts['train'][i]:3d} val={counts['val'][i]:3d}")
print("라벨/이미지 검사 통과")


## 라벨 미리보기
초록 박스가 정답 라벨입니다. LED 몸체만 포함하는지 확인하세요.
원본 파일은 변형하지 않았고, 라벨은 수동 및 색상 보조 방식으로 만든 뒤 연락판에서 시각 검토했습니다.
잘못된 박스가 보이면 학습 전에 수정해야 합니다. `label_review/`에 전체 미리보기가 있습니다.


In [ ]:
from IPython.display import display
for class_id in range(7):
    preview = sorted((DATASET / "label_review").glob(f"{class_id}_*.jpg"))[0]
    print(EXPECTED_CLASSES[class_id])
    display(Image.open(preview).resize((960, 780)))


In [ ]:
from datetime import datetime
RUN_NAME = f"yolo11{MODEL_SIZE}_components7_{datetime.now():%Y%m%d_%H%M%S}"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/component_yolo_runs")
else:
    PROJECT = Path("/content/component_yolo_runs")
    print("주의: 런타임이 종료되면 /content의 결과가 사라질 수 있습니다. 결과를 꼭 다운로드하세요.")
PROJECT.mkdir(parents=True, exist_ok=True)
print("저장 위치:", PROJECT / RUN_NAME)


## GPU 학습
기존 3클래스 모델 대신 공식 사전 학습 YOLO11 가중치에서 시작합니다.
색상별 LED 구분을 유지하려고 **색조 변형(hsv_h)은 끕니다**.
GPU 여유 메모리에 맞춰 batch를 자동으로 선택합니다. 메모리 부족 시 `batch=8` 또는 `IMGSZ=416`으로 재실행하세요.
GPU는 속도를 높여주지만 정확도를 보장하지는 않습니다.


In [ ]:
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
    batch=-1, device=0, workers=2,
    project=str(PROJECT), name=RUN_NAME, exist_ok=False,
    seed=SEED, deterministic=True,
    optimizer="AdamW", lr0=0.001, weight_decay=0.0005,
    degrees=30, translate=0.1, scale=0.3,
    fliplr=0.5, flipud=0.2,
    hsv_h=0.0, hsv_s=0.2, hsv_v=0.3,
    mosaic=0.5, close_mosaic=10, mixup=0.0,
    save=True, save_period=10, plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.is_file()
print("최종 모델:", BEST)


In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=DATA_YAML, imgsz=IMGSZ, batch=8,
                         device=0, workers=2, plots=True,
                         project=str(RUN_DIR), name="evaluation")
print("mAP@50:", metrics.box.map50)
print("mAP@50–95:", metrics.box.map)
per_class = []
for i, class_id in enumerate(metrics.box.ap_class_index):
    row = {"class": best_model.names[int(class_id)],
           "precision": float(metrics.box.p[i]), "recall": float(metrics.box.r[i]),
           "mAP50": float(metrics.box.ap50[i]), "mAP50_95": float(metrics.box.ap[i])}
    per_class.append(row)
    print(row)
summary = {"model": MODEL_WEIGHTS, "classes": best_model.names,
           "metrics": {k: float(v) for k, v in metrics.results_dict.items()},
           "per_class": per_class,
           "limitations": "Small dataset from same capture environment; no independent test set."}
(RUN_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))


In [ ]:
# 클래스별로 검증 사진을 한 장씩 표시합니다.
from collections import defaultdict
examples = {}
for item in manifest:
    if item.get("split") == "val" and item["class_id"] not in examples:
        examples[item["class_id"]] = DATASET / item["packaged_image"]
for class_id, image_path in sorted(examples.items()):
    result = best_model.predict(str(image_path), imgsz=IMGSZ, conf=0.35,
                                device=0, verbose=False)[0]
    print("정답 클래스:", EXPECTED_CLASSES[class_id])
    display(Image.fromarray(result.plot()[..., ::-1]))


In [ ]:
import shutil
# 실행 중인 기존 로컬 best.pt와 이름이 섞이지 않도록 별도 이름으로 다운로드합니다.
EXPORT = Path("/content") / f"{RUN_NAME}_best.pt"
shutil.copy2(BEST, EXPORT)
files.download(str(EXPORT))
RESULT_ZIP = shutil.make_archive(str(Path("/content") / f"{RUN_NAME}_results"),
                                "zip", root_dir=RUN_DIR)
files.download(RESULT_ZIP)
print("새 모델의 클래스:", best_model.names)


## 노트북으로 돌아와 적용
다운로드한 가중치는 이전 3클래스 `best.pt`를 덮어쓰지 말고 별도 파일로 보관하세요.
예를 들어 `~/turtlebot3_ws/yolo_training/best_components7.pt`로 저장한 뒤:

```bash
source /opt/ros/humble/setup.bash
export ROS_DOMAIN_ID=44
cd ~/turtlebot3_ws/yolo_training
.venv/bin/python infer_ros.py --ros-args \
  -p weights:=/home/kim/turtlebot3_ws/yolo_training/best_components7.pt \
  -p input_topic:=/csi/image_raw \
  -p imgsz:=320
```

다른 터미널에서 `ros2 run rqt_image_view rqt_image_view /yolo/image_raw`로 확인합니다.
640으로 학습해도 CPU 실시간 추론은 320/416으로 할 수 있지만 작은 LED 성능은 따로 확인해야 합니다.
`n`과 `s` 비교는 같은 분할과 설정으로 학습하고, 실제 노트북 CPU의 FPS도 함께 비교하세요.

### 중간에 GPU 연결이 끊겼다면
Drive에 저장된 **미완료 학습의** `weights/last.pt`로 재개할 수 있습니다.
새 런타임에서 설치/GPU 확인, 동일 ZIP 업로드/검사, Drive 마운트를 실행한 다음:

```python
checkpoint = "/content/drive/MyDrive/component_yolo_runs/원래실행폴더/weights/last.pt"
YOLO(checkpoint).train(resume=True, data=DATA_YAML, device=0)
```

런타임마다 압축 해제 위치가 달라지므로 `data=DATA_YAML`로 현재 경로를 지정하세요.
완료된 모델에 추가 학습하려면 resume 대신 `YOLO(best경로).train(data=DATA_YAML, ...)`을 사용합니다.

[Ultralytics 학습 문서](https://docs.ultralytics.com/modes/train/) ·
[Colab GPU/제한 안내](https://research.google.com/colaboratory/faq.html)
